# Verification of Generated Texts — DeepSeek API — batch_01

Runs `prompt_verification.md` on every JSON file in `prompts_batches_20/batch_01/`.

- **System prompt**: `prompt_verification.md` (the auditor instructions)
- **User message**: the JSON object `{id, specie, specie_en, deb_model, prompt, generated_text}`
- **Result saved**: key `verification` added to each JSON file (skip if already present or if `generated_text` is missing)

In [1]:
import json
import os
import time
import urllib.request
import urllib.error
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path("__file__").resolve().parents[2] / ".env") if "__file__" in dir() else load_dotenv()

# ── Configuration ─────────────────────────────────────────────────────────────
API_KEY            = os.environ.get("DEEPSEEK_API_KEY")
MODEL              = "deepseek-v4-pro"
API_URL            = "https://api.deepseek.com/chat/completions"
GENERATED_DIR      = Path("../prompts_batches_20/batch_01")
PROMPT_FILE        = Path("../verif_prompts/prompt_verification.md")

DELAY_BETWEEN_CALLS = 5
MAX_RETRIES         = 6
RETRY_DELAY         = 30
# ──────────────────────────────────────────────────────────────────────────────

if not API_KEY:
    raise ValueError("DEEPSEEK_API_KEY environment variable is not set.")

SYSTEM_PROMPT = PROMPT_FILE.read_text(encoding="utf-8")

json_files = sorted(GENERATED_DIR.glob("*.json"))
print(f"Model          : {MODEL}")
print(f"API URL        : {API_URL}")
print(f"Found          : {len(json_files)} file(s) in {GENERATED_DIR}")


Model          : deepseek-v4-pro
API URL        : https://api.deepseek.com/chat/completions
Found          : 370 file(s) in ../prompts_batches_20/batch_01


In [2]:
def call_deepseek_verify(system_prompt: str, user_content: str) -> tuple[str, dict]:
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_content},
    ]
    body = json.dumps({"model": MODEL, "messages": messages}).encode()
    req = urllib.request.Request(
        API_URL,
        data=body,
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=180) as resp:
        data = json.loads(resp.read().decode())
    return data["choices"][0]["message"]["content"], data.get("usage", {})


# ── Main loop ─────────────────────────────────────────────────────────────────
fatal = False
for idx, filepath in enumerate(json_files, 1):
    data = json.loads(filepath.read_text(encoding="utf-8"))

    if data.get("verification") is not None:
        print(f"[SKIP-verified]  {filepath.name}")
        continue

    if not data.get("generated_text"):
        print(f"[SKIP-no-text]   {filepath.name}")
        continue

    # Build the user message: only the fields the auditor needs
    user_payload = json.dumps({
        "id":             data.get("id"),
        "specie":         data.get("specie"),
        "specie_en":      data.get("specie_en"),
        "deb_model":      data.get("deb_model"),
        "prompt":         data.get("prompt"),
        "generated_text": data.get("generated_text"),
    }, ensure_ascii=False)

    print(f"[{idx:04d}/{len(json_files)}] {filepath.name} ...", end=" ", flush=True)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            raw_result, usage = call_deepseek_verify(SYSTEM_PROMPT, user_payload)

            # Parse the JSON response from DeepSeek
            try:
                verification = json.loads(raw_result)
            except json.JSONDecodeError:
                # Sometimes the model wraps the JSON in ```json ... ```
                stripped = raw_result.strip().removeprefix("```json").removesuffix("```").strip()
                verification = json.loads(stripped)

            data["verification"] = verification
            filepath.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

            verdict = verification.get("verdict", "?")
            critical = verification.get("critical_errors", "?")
            cached = usage.get("prompt_cache_hit_tokens", 0)
            print(f"{verdict} | {critical} critical | {usage.get('prompt_tokens','?')} in / {usage.get('completion_tokens','?')} out / {cached} cached")
            break

        except urllib.error.HTTPError as e:
            body_err = e.read().decode(errors="replace")
            print(f"\n  HTTP {e.code} on attempt {attempt}: {body_err[:200]}")
            if e.code == 401:
                print("  Invalid API key — check DEEPSEEK_API_KEY. Aborting.")
                fatal = True
                break
            elif e.code == 404:
                print(f"  Model '{MODEL}' not found. Aborting.")
                fatal = True
                break
            elif e.code == 403:
                print("  Access forbidden — skipping this file.")
                break
            elif e.code == 429:
                wait = RETRY_DELAY * (2 ** (attempt - 1))
                print(f"  Rate limited — waiting {wait}s before retry...", flush=True)
                time.sleep(wait)
            elif attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
        except Exception as e:
            print(f"\n  Error on attempt {attempt}: {e}")
            if attempt < MAX_RETRIES:
                time.sleep(5 * attempt)
    else:
        print(f"  FAILED after {MAX_RETRIES} attempts — skipping.")

    if fatal:
        break

    if idx < len(json_files):
        time.sleep(DELAY_BETWEEN_CALLS)

print("\nAll done.")

[0001/370] 0001_Abatus_cordatus_abj.json ... FAIL | 19 critical | 36053 in / 4781 out / 12032 cached
[0002/370] 0002_Abbottina_rivularis_abj.json ... FAIL | 2 critical | 34382 in / 1340 out / 12032 cached
[0003/370] 0003_Ablennes_hians_abj.json ... FAIL | 18 critical | 34946 in / 3235 out / 12032 cached
[0004/370] 0004_Abra_segmentum_abj.json ... FAIL | 3 critical | 32595 in / 3410 out / 12032 cached
[0005/370] 0005_Abralia_multihamata_abj.json ... FAIL | 11 critical | 33652 in / 4179 out / 12032 cached
[0006/370] 0006_Abralia_trigonura_abj.json ... FAIL | 7 critical | 32793 in / 2968 out / 12032 cached
[0007/370] 0007_Abraliopsis_morisii_abj.json ... FAIL | 10 critical | 31723 in / 4168 out / 12032 cached
[0008/370] 0008_Abramis_brama_abj.json ... FAIL | 7 critical | 36024 in / 2888 out / 12032 cached
[0009/370] 0009_Abroscopus_superciliaris_std.json ... FAIL | 5 critical | 34638 in / 1879 out / 12032 cached
[0010/370] 0010_Abrothrix_andinus_stx.json ... FAIL | 7 critical | 35805 in /

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
from collections import Counter

counts = Counter()
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    if not data.get("generated_text"):
        counts["NO_GENERATED_TEXT"] += 1
        continue
    v = data.get("verification")
    if v is None:
        counts["NOT_VERIFIED"] += 1
    else:
        counts[v.get("verdict", "UNKNOWN")] += 1

total = sum(counts.values())
print(f"=== Verification Summary ({total} files) ===")
for verdict, count in counts.most_common():
    print(f"  {verdict}: {count}")

In [ ]:
# ── Detail: FAIL files ────────────────────────────────────────────────────────
fails = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "FAIL":
        fails.append({
            "file":            filepath.name,
            "specie":          data.get("specie"),
            "critical_errors": v.get("critical_errors"),
            "error_codes":     [e.get("code") for e in v.get("errors", [])],
        })

print(f"FAIL ({len(fails)} files):")
for r in fails:
    print(f"  {r['file']}  —  {r['specie']}  —  {r['critical_errors']} critical  —  {r['error_codes']}")

In [ ]:
# ── Detail: GENERATION_FAILURE files ─────────────────────────────────────────
failures = []
for filepath in sorted(GENERATED_DIR.glob("*.json")):
    data = json.loads(filepath.read_text(encoding="utf-8"))
    v = data.get("verification")
    if v and v.get("verdict") == "GENERATION_FAILURE":
        failures.append({"file": filepath.name, "specie": data.get("specie")})

print(f"GENERATION_FAILURE ({len(failures)} files):")
for r in failures:
    print(f"  {r['file']}  —  {r['specie']}")